# Cats vs Dogs, with Help from a Pretrained Image Classification Model

<img src="https://raw.githubusercontent.com/benjum/UCLAX-ML/main/Data/cat-dog-data/test/cat/cat.1500.jpg" width="250px"/> <img src="https://raw.githubusercontent.com/benjum/UCLAX-ML/main/Data/cat-dog-data/test/dog/dog.1500.jpg" width="250px"/>

#### We're going to use PyTorch and a pretrained convolutional neural network to identify whether a picture is a dog or a cat.

Side-note: I think the dataset is biased against cats, because most of the cats I've looked at in this dataset look crazy, a bit ugly, or like they totally have it out for the dogs!

This dataset is a subset of the data that can be obtained at [Kaggle's Dogs Vs Cats page](https://www.kaggle.com/c/dogs-vs-cats/data): 2,000 training images, 1,000 validation, and 2,000 test, half cats and half dogs.

With only 2,000 training images, a CNN trained from scratch overfits quickly. Instead, we start from **VGG16**, a CNN that was already trained on **ImageNet**: 1.2 million photos in 1,000 categories, including many breeds of cats and dogs. Its convolutional layers have already learned to detect edges, textures, fur, eyes, and ears. Reusing a network trained on one task for another is called **transfer learning**. We'll do it in three ways:

1. **Feature extraction**: run our images through VGG16's convolutional layers once, and train a small classifier on the resulting features. Fast, even on a CPU.
2. **Feature extraction with data augmentation**: keep VGG16's layers frozen, but train the whole model end to end on randomly flipped, rotated, and zoomed images.
3. **Fine-tuning**: also adjust VGG16's last few layers, very gently, for our task.

The approach follows Chollet's *Deep Learning with Python*, in which you will find many, many more interesting details and tidbits about doing deep learning.

In [ ]:
import copy
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms.v2 as T
from torch.utils.data import TensorDataset, DataLoader, Subset

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("PyTorch", torch.__version__, "  torchvision", torchvision.__version__, "  device:", device)

Special note: parts 2 and 3 below run the full VGG16 network on every training image in every epoch. That takes a long time on a CPU, so a GPU is highly recommended for them. If you need help running this notebook on a GPU, please reach out for assistance.

To check that everything works before a long run, set `QUICK_TEST = True`: every part then runs for 1 epoch on 64 images per set. (The results will be meaningless, but any errors will show up within a minute or two.)

In [ ]:
QUICK_TEST = False

EPOCHS_FEATURES = 1 if QUICK_TEST else 20
EPOCHS_AUGMENTATION = 1 if QUICK_TEST else 50
EPOCHS_FINE_TUNING = 1 if QUICK_TEST else 30
NUM_WORKERS = 4 if device == "cuda" else 0     # background processes that load images during training

## Getting our data

Our data images are in folders that are already split up into training, test, and validation sets. Furthermore, they are already split up into cat and dog folders.

`torchvision.datasets.ImageFolder` gets the class labels from the folder names: each subfolder becomes a class, numbered in alphabetical order (`cat` = 0, `dog` = 1).

In [ ]:
DATA_DIR = Path("..", "..", "Data", "cat-dog-data")
print(DATA_DIR.resolve(), DATA_DIR.is_dir())

VGG16 expects its input prepared the same way as the images it was trained on. Each image is:
1. resized to 180×180 pixels (the photos all have different sizes)
2. converted to a tensor of floats between 0 and 1, with shape `(3, 180, 180)`: 3 color channels, then height and width
3. **normalized** with the mean and standard deviation of each color channel over ImageNet's images

In [ ]:
IMAGE_SIZE = 180
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

preprocess = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE), antialias=True),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

def image_folder(split, transform):
    dataset = torchvision.datasets.ImageFolder(DATA_DIR / split, transform=transform)
    if QUICK_TEST:   # 32 cats (the first images) and 32 dogs (the last images)
        dataset = Subset(dataset, list(range(32)) + list(range(len(dataset) - 32, len(dataset))))
    return dataset

train_dataset = image_folder("train", preprocess)
validation_dataset = image_folder("validation", preprocess)
test_dataset = image_folder("test", preprocess)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=NUM_WORKERS)
validation_loader = DataLoader(validation_dataset, batch_size=32, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=32, num_workers=NUM_WORKERS)

print(len(train_dataset), "training images,", len(validation_dataset), "validation,", len(test_dataset), "test")

Each example is a pair `(image, label)`:

In [ ]:
image, label = train_dataset[2]
print("image shape:", image.shape, "  label:", label)
print("a few pixel values:", image[0, 2, :5])

In [ ]:
raw = torchvision.datasets.ImageFolder(DATA_DIR / "train")    # no transform: the original photos

b = 2
picture, label = raw[b]
print("The label for image", b, "is", label, f"({raw.classes[label]})")
print("The picture for image", b, "is")
plt.imshow(picture)
plt.axis("off")
plt.show()

# Using a pretrained model

`torchvision.models` provides many models with pretrained weights. "These models can be used for prediction, feature extraction, and fine-tuning."
* https://pytorch.org/vision/stable/models.html

The first time, this downloads VGG16's weights (about 530 MB).

In [ ]:
vgg16 = torchvision.models.vgg16(weights=torchvision.models.VGG16_Weights.IMAGENET1K_V1)
vgg16

The model has two main parts:
* **`features`**: the convolutional base, 13 convolutional layers in 5 blocks, each block ending with max pooling
* **`avgpool`** and **`classifier`**: dense layers that output scores for ImageNet's 1,000 categories

We keep only the convolutional base, `vgg16.features`. (Keras calls this `include_top=False`.) Its 5 pooling layers shrink our 180×180 images to 5×5, with 512 channels:

In [ ]:
conv_base = vgg16.features.to(device)
conv_base.eval()     # no dropout or batch norm inside, but a good habit

with torch.no_grad():
    print("output shape:", conv_base(torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)).shape)
print(f"parameters in the convolutional base: {sum(p.numel() for p in conv_base.parameters()):,}")

Training functions for binary classification, used for all three parts. As in the review notebook, the model outputs **one logit** and the loss is `nn.BCEWithLogitsLoss`, so the labels must be floats with shape `(batch, 1)`.

`fit` also keeps a copy of the parameters from the epoch with the **lowest validation loss**, and restores them at the end. (Keras's `ModelCheckpoint(save_best_only=True, monitor="val_loss")` does the same, saving to a file instead.)

In [ ]:
def run_epoch(model, loader, loss_fn, optimizer=None):
    # Training if an optimizer is given, otherwise evaluation. Returns (loss, accuracy).
    training = optimizer is not None
    model.train(training)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.float().reshape(-1, 1).to(device)
            logits = model(inputs)
            loss = loss_fn(logits, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            correct += ((logits > 0).float() == labels).sum().item()
            n += len(labels)
    return total_loss / n, correct / n


def fit(model, train_loader, validation_loader, optimizer, epochs):
    loss_fn = nn.BCEWithLogitsLoss()
    history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
    best_val_loss, best_state = float("inf"), None
    for epoch in range(epochs):
        loss, accuracy = run_epoch(model, train_loader, loss_fn, optimizer)
        val_loss, val_accuracy = run_epoch(model, validation_loader, loss_fn)
        for key, value in zip(history, (loss, accuracy, val_loss, val_accuracy)):
            history[key].append(value)
        if val_loss < best_val_loss:
            best_val_loss, best_state = val_loss, copy.deepcopy(model.state_dict())
        print(f"Epoch {epoch + 1}/{epochs}   loss: {loss:.4f}  accuracy: {accuracy:.4f}"
              f"   val_loss: {val_loss:.4f}  val_accuracy: {val_accuracy:.4f}")
    model.load_state_dict(best_state)     # this loads the best weights into the model
    return history


def plot_history(history):
    epochs = range(1, len(history["accuracy"]) + 1)
    plt.plot(epochs, history["accuracy"], "bo", label="Training accuracy")
    plt.plot(epochs, history["val_accuracy"], "b", label="Validation accuracy")
    plt.title("Training and validation accuracy")
    plt.legend()
    plt.figure()
    plt.plot(epochs, history["loss"], "bo", label="Training loss")
    plt.plot(epochs, history["val_loss"], "b", label="Validation loss")
    plt.title("Training and validation loss")
    plt.legend()
    plt.show()


def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 1. Feature extraction without data augmentation

Run every image through the convolutional base **once**, and store the outputs (the "features"). Then train a small dense classifier on those features. The expensive part, VGG16, runs only once per image, so the training itself is fast.

In [ ]:
def get_features_and_labels(loader):
    all_features, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            features = conv_base(images.to(device))
            all_features.append(features.cpu())
            all_labels.append(labels)
    return torch.cat(all_features), torch.cat(all_labels)

train_features, train_labels = get_features_and_labels(train_loader)
val_features, val_labels = get_features_and_labels(validation_loader)
test_features, test_labels = get_features_and_labels(test_loader)

In [ ]:
train_features.shape

The classifier: flatten the 512 × 5 × 5 features, a dense layer of 256 units, dropout, and one output (the logit).

In [ ]:
def make_classifier():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(512 * 5 * 5, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, 1),
    )

torch.manual_seed(42)
model = make_classifier().to(device)

feature_train_loader = DataLoader(TensorDataset(train_features, train_labels), batch_size=32, shuffle=True)
feature_val_loader = DataLoader(TensorDataset(val_features, val_labels), batch_size=32)
feature_test_loader = DataLoader(TensorDataset(test_features, test_labels), batch_size=32)

optimizer = torch.optim.RMSprop(model.parameters(), lr=1e-3)
history = fit(model, feature_train_loader, feature_val_loader, optimizer, epochs=EPOCHS_FEATURES)

In [ ]:
plot_history(history)

In [ ]:
test_loss, test_acc = run_epoch(model, feature_test_loader, nn.BCEWithLogitsLoss())
print(f"Test accuracy: {test_acc:.3f}")

This should be much better than a CNN trained from scratch on 2,000 images. But the curves typically show the classifier overfitting within a few epochs: the training accuracy approaches 100% while the validation accuracy levels off. We can't use data augmentation here, because the features were computed once, from the original images.

## 2. Feature extraction with data augmentation

Now the convolutional base becomes part of the model, so every training image goes through it in every epoch, each time randomly modified. To keep what VGG16 learned on ImageNet, we **freeze** the convolutional base: setting `requires_grad = False` on its parameters means no gradients are computed for them, and the optimizer leaves them unchanged.

In [ ]:
class CatDogTransfer(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_base = torchvision.models.vgg16(weights=torchvision.models.VGG16_Weights.IMAGENET1K_V1).features
        self.classifier = make_classifier()

    def forward(self, x):
        return self.classifier(self.conv_base(x))

torch.manual_seed(42)
model = CatDogTransfer().to(device)

print(f"trainable parameters, everything: {count_trainable(model):,}")
for p in model.conv_base.parameters():
    p.requires_grad = False
print(f"trainable parameters, conv base frozen: {count_trainable(model):,}")

Data augmentation is applied only to the **training** images, as part of their transform, before the normalization:
* `RandomHorizontalFlip`: mirror the image half of the time
* `RandomRotation(36)`: rotate by up to ±36° (a tenth of a full turn)
* `RandomAffine(..., scale=(0.8, 1.2))`: zoom in or out by up to 20%

In [ ]:
augment = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE), antialias=True),
    T.RandomHorizontalFlip(),
    T.RandomRotation(36),
    T.RandomAffine(degrees=0, scale=(0.8, 1.2)),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

augmented_train_dataset = image_folder("train", augment)
augmented_train_loader = DataLoader(augmented_train_dataset, batch_size=32, shuffle=True, num_workers=NUM_WORKERS)

torch.manual_seed(1)
picture = raw[2][0]
fig, axes = plt.subplots(1, 6, figsize=(14, 2.6))
axes[0].imshow(picture.resize((IMAGE_SIZE, IMAGE_SIZE)))
axes[0].set_title("original")
for ax in axes[1:]:
    augmented = augment(picture)
    shown = augmented * torch.tensor(IMAGENET_STD).reshape(3, 1, 1) + torch.tensor(IMAGENET_MEAN).reshape(3, 1, 1)
    ax.imshow(shown.clamp(0, 1).permute(1, 2, 0))     # undo the normalization to display it
    ax.set_title("augmented")
for ax in axes:
    ax.axis("off")
plt.show()

The optimizer only receives the parameters that are still trainable:

In [ ]:
optimizer = torch.optim.RMSprop([p for p in model.parameters() if p.requires_grad], lr=1e-3)
history = fit(model, augmented_train_loader, validation_loader, optimizer, epochs=EPOCHS_AUGMENTATION)

In [ ]:
plot_history(history)

In [ ]:
test_loss, test_acc = run_epoch(model, test_loader, nn.BCEWithLogitsLoss())
print(f"Test accuracy: {test_acc:.3f}")

## 3. Fine-tuning

Finally, we **unfreeze the last convolutional block** of VGG16 (its last 3 convolutional layers) and train them together with the classifier. These top layers detect the most complex, most task-specific patterns, so adjusting them for cats and dogs can help. The earlier layers detect general patterns, such as edges and textures, and stay frozen.

Two important details:
* Fine-tune only **after** the classifier has been trained (part 2). A randomly initialized classifier would send large, random gradients into VGG16 and destroy what it learned.
* Use a **very small learning rate** (here $10^{-5}$), so the pretrained weights change only gently.

In `conv_base`, the last block starts at index 24:

In [ ]:
for index, layer in enumerate(model.conv_base):
    print(index, layer)

In [ ]:
for layer in model.conv_base[24:]:
    for p in layer.parameters():
        p.requires_grad = True
print(f"trainable parameters: {count_trainable(model):,}")

optimizer = torch.optim.RMSprop([p for p in model.parameters() if p.requires_grad], lr=1e-5)
history = fit(model, augmented_train_loader, validation_loader, optimizer, epochs=EPOCHS_FINE_TUNING)

In [ ]:
plot_history(history)

In [ ]:
test_loss, test_acc = run_epoch(model, test_loader, nn.BCEWithLogitsLoss())
print(f"Test accuracy: {test_acc:.3f}")

## Summary

* **Transfer learning**: a network pretrained on a large dataset (here, VGG16 on ImageNet) already detects useful visual patterns, so a small dataset is enough to adapt it to a new task.
* **Feature extraction**: run the images through the frozen convolutional base once, and train a small classifier on the features. Fast, but no data augmentation is possible.
* **Feature extraction with data augmentation**: put the frozen base inside the model and train end to end on augmented images. Slower, but overfits less.
* **Fine-tuning**: after the classifier is trained, unfreeze the top layers of the base and train with a very small learning rate.
* In PyTorch, **freezing** means setting `requires_grad = False` on parameters, and giving the optimizer only the trainable ones.